# 从零实现 GCN：归一化、训练与可验证制品

本 Notebook 只用 PyTorch 张量和 `nn.Module` 手写两层 GCN，不使用 PyG、DGL、`torch_geometric` 或任何现成 GNN 层。链路覆盖：tenant 前置过滤 → 受控小图 → 手写 `A+I` 对称归一化 → `GCNLayer/GCNNet` → mask 半监督训练 → NumPy 对照 → 梯度检查 → 制品/图/特征绑定 → 推理权限。

所有数据均为固定种子的虚构服务节点，强制 CPU 和单线程，结果用于验证公式与工程合同，不代表真实数据效果或大图吞吐。

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

from dataclasses import dataclass
import hashlib
import json
import math
import random
import time

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 2501
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
DTYPE = torch.float32

def canonical_fingerprint(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

assert torch.__version__ and DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
assert torch.initial_seed() == SEED

## 1. 数据合同：先隔离 tenant，再构造邻接

tenant-a 含 24 个节点、两类各 12 个；同类节点形成环和二跳边，只有少量跨类边。tenant-b 使用极端特征，并故意放入一条跨 tenant 恶意边。授权函数必须先按可信上下文过滤节点和边，再建立局部索引；若先在全图归一化后裁剪，tenant-b 会改变 tenant-a 节点的度数与消息尺度。

In [ ]:
@dataclass(frozen=True)
class AuthContext:
    tenant: str
    scopes: frozenset[str]
    principal: str
    def require(self, scope: str) -> None:
        if scope not in self.scopes:
            raise PermissionError(f"缺少 scope: {scope}")

auth_a = AuthContext("tenant-a", frozenset({"graph:read", "model:predict"}), "alice")
node_ids_all = [f"tenant-a:n{i:02d}" for i in range(24)] + ["tenant-b:x0", "tenant-b:x1"]
tenant_all = ["tenant-a"] * 24 + ["tenant-b"] * 2

generator = torch.Generator(device="cpu").manual_seed(SEED)
y_all = torch.tensor([0] * 12 + [1] * 12 + [-1, -1], dtype=torch.long)
base0 = torch.tensor([1.2, 0.2, 0.5, 0.1, 1.0])
base1 = torch.tensor([0.2, 1.2, 0.3, 0.7, 1.0])
X_a = torch.stack([(base0 if i < 12 else base1) + 0.22 * torch.randn(5, generator=generator) * torch.tensor([1,1,1,1,0]) for i in range(24)])
X_all = torch.cat([X_a, torch.tensor([[50.,50.,50.,50.,1.],[-50.,-50.,-50.,-50.,1.]])], dim=0).to(DTYPE)

within_pairs = set()
for offset in (0, 12):
    for j in range(12):
        for step in (1, 2):
            within_pairs.add(tuple(sorted((offset + j, offset + (j + step) % 12))))
raw_pairs = sorted(within_pairs | {(0,12), (4,16), (8,20), (24,25), (0,24)})

def authorized_graph(auth: AuthContext):
    auth.require("graph:read")
    visible = [i for i, tenant in enumerate(tenant_all) if tenant == auth.tenant]
    remap = {old: new for new, old in enumerate(visible)}
    pairs = [(remap[u], remap[v]) for u, v in raw_pairs if u in remap and v in remap and tenant_all[u] == tenant_all[v] == auth.tenant]
    return [node_ids_all[i] for i in visible], X_all[visible].clone(), y_all[visible].clone(), sorted(pairs)

node_ids, X, y, edge_pairs = authorized_graph(auth_a)
assert len(node_ids) == X.shape[0] == y.numel() == 24
assert X.shape == (24, 5) and set(y.tolist()) == {0, 1}
assert all(n.startswith("tenant-a:") for n in node_ids)
assert (0, 24) not in edge_pairs and len(edge_pairs) == 51

## 2. Mask 与 transductive 边界

每类使用 6 个 train、3 个 validation、3 个 test。GCN 前向会使用授权快照中所有节点的**无标签特征和结构**，所以这是 transductive 半监督设定；只有 train 标签进入损失，validation 只选 checkpoint，test 在选型后只报告一次。若目标是 inductive，新节点及其边不能参与训练期传播，应使用 GraphSAGE 等可复用聚合器并重新定义评估协议。

In [ ]:
train_mask = torch.zeros(24, dtype=torch.bool)
val_mask = torch.zeros(24, dtype=torch.bool)
test_mask = torch.zeros(24, dtype=torch.bool)
for cls in (0, 1):
    idx = torch.where(y == cls)[0]
    train_mask[idx[:6]] = True
    val_mask[idx[6:9]] = True
    test_mask[idx[9:12]] = True

coverage = train_mask.to(torch.int8) + val_mask.to(torch.int8) + test_mask.to(torch.int8)
assert (int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum())) == (12, 6, 6)
assert torch.equal(coverage, torch.ones_like(coverage))
assert set(y[train_mask].tolist()) == set(y[val_mask].tolist()) == set(y[test_mask].tolist()) == {0, 1}
assert not torch.any(train_mask & val_mask) and not torch.any(train_mask & test_mask)

## 3. 手写 GCN 归一化

对无向邻接矩阵 (A\in\mathbb{R}^{N\times N})，先加自环 (	ilde A=A+I)，再令 (	ilde D_{ii}=\sum_j	ilde A_{ij})：

[
hat A=	ilde D^{-1/2}	ilde A	ilde D^{-1/2},\qquad H^{(l+1)}=\sigma(hat A H^{(l)}W^{(l)}+b^{(l)})
]

输入 `H:(N,F_in)`、权重 `W:(F_in,F_out)`，输出为 `(N,F_out)`。对称归一化通常不逐行和为 1；孤立点加自环后度数为 1，不会除零。

仅检查“大图输出对称、有限”还不足以锁定公式：错误的度数指数也可能保持对称。下面增加一个独立的 3-node path + 1 isolated node oracle。前三个节点的度数（加自环后）应为 `[2,3,2]`，边权精确为 $1/\sqrt{6}$；孤点加自环后整行只在对角线为 1。该 fixture 同时固定自环时机、归一化指数与孤点行为。


In [ ]:
def dense_adjacency(num_nodes: int, pairs: list[tuple[int, int]]) -> torch.Tensor:
    if num_nodes <= 0:
        raise ValueError("num_nodes 必须为正")
    A = torch.zeros((num_nodes, num_nodes), dtype=DTYPE, device=DEVICE)
    for u, v in pairs:
        if u == v or not (0 <= u < num_nodes and 0 <= v < num_nodes):
            raise ValueError("边端点非法；自环由归一化函数统一加入")
        A[u, v] = 1.0; A[v, u] = 1.0
    return A

def gcn_normalize(A: torch.Tensor, add_self_loops: bool = True) -> torch.Tensor:
    if A.ndim != 2 or A.shape[0] != A.shape[1] or not torch.allclose(A, A.T):
        raise ValueError("GCN 示例要求方形对称邻接")
    if not torch.isfinite(A).all() or (A < 0).any():
        raise ValueError("邻接权重必须有限且非负")
    A_tilde = A + torch.eye(A.shape[0], dtype=A.dtype, device=A.device) if add_self_loops else A.clone()
    degree = A_tilde.sum(dim=1)
    if (degree <= 0).any():
        raise ValueError("存在零度节点；请加自环或定义 isolated fallback")
    inv_sqrt = degree.rsqrt()
    return inv_sqrt[:, None] * A_tilde * inv_sqrt[None, :]

A = dense_adjacency(len(node_ids), edge_pairs)
A_hat = gcn_normalize(A)
assert A.shape == A_hat.shape == (24, 24)
assert torch.allclose(A_hat, A_hat.T, atol=1e-7)
assert torch.all(torch.diag(A_hat) > 0) and torch.isfinite(A_hat).all()
assert not torch.allclose(A_hat.sum(1), torch.ones(24), atol=1e-4)
# 独立小图 oracle：0--1--2，节点 3 完全孤立。
oracle_A = dense_adjacency(4, [(0, 1), (1, 2)])
oracle_hat = gcn_normalize(oracle_A, add_self_loops=True)
oracle_expected = torch.zeros((4, 4), dtype=DTYPE)
oracle_expected[0, 0] = 1 / 2
oracle_expected[1, 1] = 1 / 3
oracle_expected[2, 2] = 1 / 2
oracle_expected[3, 3] = 1
oracle_expected[0, 1] = oracle_expected[1, 0] = 1 / math.sqrt(6)
oracle_expected[1, 2] = oracle_expected[2, 1] = 1 / math.sqrt(6)
assert torch.allclose(oracle_hat, oracle_expected, atol=1e-7)
assert torch.equal(oracle_hat[3], torch.tensor([0., 0., 0., 1.]))
assert torch.allclose(oracle_hat.diag(), torch.tensor([0.5, 1/3, 0.5, 1.0]), atol=1e-7)
try:
    gcn_normalize(oracle_A, add_self_loops=False)
    raise AssertionError("关闭自环时孤点没有 fail-closed")
except ValueError as exc:
    assert "零度节点" in str(exc)


## 4. `GCNLayer`：参数、shape 与 forward

本层显式注册 (W) 和 (b)，不用 `nn.Linear` 隐藏矩阵次序。先计算 `support=X@W`，再左乘 `Â`；由于矩阵乘法结合律，也可先聚合再线性变换。参数量为 `F_in*F_out + F_out`。forward 会拒绝维度、设备和非有限输入，避免广播产生“能跑但语义错”的结果。

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        if in_features <= 0 or out_features <= 0:
            raise ValueError("特征维度必须为正")
        self.in_features, self.out_features = in_features, out_features
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x: torch.Tensor, adj_norm: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[1] != self.in_features:
            raise ValueError("x shape 与 in_features 不匹配")
        if adj_norm.shape != (x.shape[0], x.shape[0]):
            raise ValueError("adj_norm 必须是与节点数一致的方阵")
        if x.device != adj_norm.device or not torch.isfinite(x).all() or not torch.isfinite(adj_norm).all():
            raise ValueError("设备不一致或存在非有限输入")
        out = adj_norm @ (x @ self.weight)
        return out + self.bias if self.bias is not None else out

layer_probe = GCNLayer(5, 7).to(DEVICE)
probe_out = layer_probe(X, A_hat)
assert probe_out.shape == (24, 7)
assert sum(p.numel() for p in layer_probe.parameters()) == 5 * 7 + 7
assert isinstance(layer_probe.weight, nn.Parameter)
assert torch.isfinite(probe_out).all()

## 5. 两层 `GCNNet`

第一层将 5 维输入映射到 8 维并使用 ReLU/Dropout，第二层输出 2 类 logits。训练时 Dropout 生效，评估时必须调用 `eval()`。模型返回未做 softmax 的 logits，因为 `cross_entropy` 内部使用数值稳定的 log-softmax。

In [ ]:
class GCNNet(nn.Module):
    def __init__(self, in_features: int, hidden: int, classes: int, dropout: float = 0.15):
        super().__init__()
        self.conv1 = GCNLayer(in_features, hidden)
        self.conv2 = GCNLayer(hidden, classes)
        self.dropout = float(dropout)

    def forward(self, x: torch.Tensor, adj_norm: torch.Tensor, return_hidden: bool = False):
        hidden = F.relu(self.conv1(x, adj_norm))
        hidden_drop = F.dropout(hidden, p=self.dropout, training=self.training)
        logits = self.conv2(hidden_drop, adj_norm)
        return (logits, hidden) if return_hidden else logits

model = GCNNet(5, 8, 2).to(DEVICE)
model.eval()
logits, hidden = model(X, A_hat, return_hidden=True)
expected_params = (5 * 8 + 8) + (8 * 2 + 2)
assert logits.shape == (24, 2) and hidden.shape == (24, 8)
assert sum(p.numel() for p in model.parameters()) == expected_params == 66
assert torch.isfinite(logits).all()
assert model(X, A_hat).device.type == "cpu"

## 6. NumPy 数值对照

框架内实现若把乘法顺序、转置或 bias 位置写错，训练仍可能在小数据上“看起来有效”。下面固定一层参数，将 PyTorch forward 与独立 NumPy 公式 `Â @ (X @ W) + b` 对照；这是公式级回归，不是与另一个 GNN 库互相复制。

In [ ]:
reference_layer = GCNLayer(5, 3)
with torch.no_grad():
    reference_layer.weight.copy_(torch.linspace(-0.4, 0.5, 15).reshape(5, 3))
    reference_layer.bias.copy_(torch.tensor([0.1, -0.2, 0.05]))
torch_value = reference_layer(X, A_hat).detach().numpy()
numpy_value = A_hat.numpy() @ (X.numpy() @ reference_layer.weight.detach().numpy()) + reference_layer.bias.detach().numpy()
max_abs_error = float(np.max(np.abs(torch_value - numpy_value)))
assert max_abs_error < 2e-6
assert np.isfinite(numpy_value).all()
assert numpy_value.shape == (24, 3)

## 7. 只用 train mask 训练，validation 选 checkpoint

每轮前向仍覆盖整个授权图，这是已披露的 transductive 传播；监督损失严格切到 `train_mask`。validation loss 只用于选择 checkpoint，不做反向传播。保存的是参数张量副本而不是可变引用；恢复最佳状态后才首次计算 test 指标。

此外，`best_state is not None` 只说明循环跑过，不能证明优化器真正更新。训练前保存独立参数副本和 eval-mode train loss；恢复最佳 checkpoint 后要求至少一个参数发生变化、train loss 明显下降且 train-mask accuracy 达标。这样即使误删 `optimizer.step()`，Notebook 也会失败。


In [ ]:
torch.manual_seed(SEED)
model = GCNNet(5, 8, 2, dropout=0.15).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.04, weight_decay=5e-4)
initial_train_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.eval()
with torch.no_grad():
    initial_train_eval_loss = float(F.cross_entropy(model(X, A_hat)[train_mask], y[train_mask]))

best_val, best_epoch, best_state = math.inf, -1, None
history = []
started = time.perf_counter()
for epoch in range(120):
    model.train(); optimizer.zero_grad(set_to_none=True)
    train_logits = model(X, A_hat)
    loss = F.cross_entropy(train_logits[train_mask], y[train_mask])
    loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        val_logits = model(X, A_hat)
        val_loss = float(F.cross_entropy(val_logits[val_mask], y[val_mask]))
    history.append((float(loss.detach()), val_loss))
    if val_loss < best_val:
        best_val, best_epoch = val_loss, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
train_seconds = time.perf_counter() - started

assert best_state is not None and 0 <= best_epoch < 120
assert len(history) == 120 and np.isfinite(np.asarray(history)).all()
assert train_seconds < 20
model.load_state_dict(best_state, strict=True); model.eval()
with torch.no_grad():
    restored_train_logits = model(X, A_hat)
    restored_train_eval_loss = float(F.cross_entropy(restored_train_logits[train_mask], y[train_mask]))
    restored_train_accuracy = float((restored_train_logits[train_mask].argmax(1) == y[train_mask]).float().mean())
updated_parameter_keys = [key for key, value in model.state_dict().items()
                          if not torch.equal(value.detach().cpu(), initial_train_state[key])]
assert updated_parameter_keys, "optimizer 没有改变任何 state_dict 张量"
assert restored_train_eval_loss < initial_train_eval_loss * 0.75
assert restored_train_accuracy >= 0.90


## 8. 梯度不是“loss 能下降”的替代品

训练代码可能因 `detach`、错误索引或未注册参数导致部分层没有梯度。恢复最佳参数后再执行一次纯 train-mask backward，检查每个可训练参数都有有限梯度，且全模型梯度范数非零；这不会调用 `optimizer.step()`，所以不会改变制品参数。

In [ ]:
model.train(); optimizer.zero_grad(set_to_none=True)
gradient_loss = F.cross_entropy(model(X, A_hat)[train_mask], y[train_mask])
gradient_loss.backward()
grad_norms = {name: float(param.grad.norm()) for name, param in model.named_parameters() if param.requires_grad}
assert set(grad_norms) == {name for name, p in model.named_parameters() if p.requires_grad}
assert all(math.isfinite(value) for value in grad_norms.values())
assert sum(grad_norms.values()) > 0
assert all(value > 0 for value in grad_norms.values())
model.eval()

## 9. 测试指标与 tenant 泄漏反例

准确率只用于受控平衡数据的流水线检查。更重要的反例是：若把恶意跨 tenant 边放入全图先归一化，再裁剪左上角，tenant-a 的第 0 个节点度数已被 tenant-b 改变。即便最终不返回 tenant-b 节点，消息尺度仍泄漏，因此 ACL 必须在归一化之前。

In [ ]:
with torch.no_grad():
    final_logits = model(X, A_hat)
    final_pred = final_logits.argmax(dim=1)
test_accuracy = float((final_pred[test_mask] == y[test_mask]).float().mean())
val_accuracy = float((final_pred[val_mask] == y[val_mask]).float().mean())

A_all_unsafe = dense_adjacency(26, raw_pairs)
A_all_hat_unsafe = gcn_normalize(A_all_unsafe)
unsafe_slice = A_all_hat_unsafe[:24, :24]
assert 0.0 <= test_accuracy <= 1.0 and 0.0 <= val_accuracy <= 1.0
assert not torch.allclose(unsafe_slice, A_hat)
assert A_all_hat_unsafe[0, 24] > 0
assert A_hat.shape[0] < A_all_hat_unsafe.shape[0]

## 10. Artifact：绑定图指纹、特征 schema 与 `state_dict`

仅保存权重文件不够：相同 shape 的模型可能配错节点顺序、图快照或特征列。图指纹使用稳定节点 ID 与无向边；特征 schema 记录顺序、dtype 和来源；`state_dict` 指纹按 key、shape、dtype 和原始字节计算。内容哈希可发现意外错配，但不能代替生产中的签名、权限和不可变存储。

这里进一步绑定**有序特征快照**，而不只绑定列 schema。快照描述包含 `snapshot_id/as_of/node_order/shape/dtype/content_sha256`；服务会从当前实际 `X` 重新计算描述再校验。于是拓扑和 schema 不变、但任意一个特征值变化时也会 fail-closed。


In [ ]:
FEATURE_SCHEMA = {"order": ["http_ratio","batch_ratio","cpu_norm","error_ratio","bias"],
                  "dtype": "float32", "source": "synthetic-v1", "fit_split": "not_applicable"}
FEATURE_SCHEMA_ID = canonical_fingerprint(FEATURE_SCHEMA)
FEATURE_SNAPSHOT_ID = "tenant-a-features-2026-07-01T00:00:00Z"
FEATURE_SNAPSHOT_AS_OF = "2026-07-01T00:00:00Z"

def feature_snapshot_descriptor(ordered_node_ids, features, snapshot_id: str, as_of: str) -> dict:
    if features.ndim != 2 or len(ordered_node_ids) != features.shape[0]:
        raise ValueError("节点顺序与特征 shape 不匹配")
    if len(set(ordered_node_ids)) != len(ordered_node_ids) or not torch.isfinite(features).all():
        raise ValueError("节点 ID 必须唯一且特征必须有限")
    value = features.detach().cpu().contiguous()
    return {
        "snapshot_id": snapshot_id,
        "as_of": as_of,
        "node_order": list(ordered_node_ids),
        "shape": list(value.shape),
        "dtype": str(value.dtype),
        "content_sha256": hashlib.sha256(value.numpy().tobytes()).hexdigest(),
    }

FEATURE_SNAPSHOT = feature_snapshot_descriptor(
    node_ids, X, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF
)
FEATURE_SNAPSHOT_FINGERPRINT = canonical_fingerprint(FEATURE_SNAPSHOT)
graph_payload = {"tenant": auth_a.tenant, "as_of": "2026-07-01Z", "nodes": node_ids,
                 "edges": sorted([sorted((node_ids[u], node_ids[v])) for u, v in edge_pairs]),
                 "normalization": "A-plus-I-symmetric-D-minus-half-v1"}
GRAPH_FINGERPRINT = canonical_fingerprint(graph_payload)

def state_dict_fingerprint(state_dict: dict[str, torch.Tensor]) -> str:
    digest = hashlib.sha256()
    for key in sorted(state_dict):
        tensor = state_dict[key].detach().cpu().contiguous()
        digest.update(key.encode()); digest.update(str(tensor.dtype).encode())
        digest.update(json.dumps(list(tensor.shape)).encode()); digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()[:20]

artifact = {
    "model_type": "GCNNet-from-scratch", "model_version": "gcn-v2",
    "tenant": auth_a.tenant, "as_of": graph_payload["as_of"],
    "graph_fingerprint": GRAPH_FINGERPRINT, "feature_schema_id": FEATURE_SCHEMA_ID,
    "feature_snapshot_id": FEATURE_SNAPSHOT_ID,
    "feature_snapshot_fingerprint": FEATURE_SNAPSHOT_FINGERPRINT,
    "node_order": node_ids, "dims": [5,8,2], "dropout": 0.15,
    "normalization": graph_payload["normalization"],
    "state_dict_fingerprint": state_dict_fingerprint(model.state_dict()), "seed": SEED,
}
artifact["artifact_id"] = canonical_fingerprint(artifact)

def validate_artifact(value: dict, current_state, current_graph_fp: str,
                      current_feature_schema_id: str, current_feature_snapshot: dict) -> bool:
    unsigned = {k: v for k, v in value.items() if k != "artifact_id"}
    if canonical_fingerprint(unsigned) != value.get("artifact_id"):
        raise ValueError("artifact 内容哈希不匹配")
    if value.get("graph_fingerprint") != current_graph_fp:
        raise ValueError("图快照与制品不匹配")
    if value.get("feature_schema_id") != current_feature_schema_id:
        raise ValueError("特征 schema 与制品不匹配")
    current_feature_fp = canonical_fingerprint(current_feature_snapshot)
    if (value.get("feature_snapshot_id") != current_feature_snapshot.get("snapshot_id") or
            value.get("feature_snapshot_fingerprint") != current_feature_fp):
        raise ValueError("有序特征快照与制品不匹配")
    if value.get("state_dict_fingerprint") != state_dict_fingerprint(current_state):
        raise ValueError("state_dict 与制品不匹配")
    return True

assert validate_artifact(artifact, model.state_dict(), GRAPH_FINGERPRINT,
                         FEATURE_SCHEMA_ID, FEATURE_SNAPSHOT)
assert len(artifact["artifact_id"]) == len(artifact["state_dict_fingerprint"]) == 20
for field, bad in (("graph_fingerprint","bad-graph"),
                   ("feature_schema_id","bad-schema"),
                   ("feature_snapshot_fingerprint","bad-feature-snapshot"),
                   ("state_dict_fingerprint","bad-state")):
    forged = dict(artifact); forged[field] = bad
    forged["artifact_id"] = canonical_fingerprint({k:v for k,v in forged.items() if k != "artifact_id"})
    try:
        validate_artifact(forged, model.state_dict(), GRAPH_FINGERPRINT,
                          FEATURE_SCHEMA_ID, FEATURE_SNAPSHOT)
        raise AssertionError(f"伪造 {field} 未被拒绝")
    except ValueError:
        pass

changed_X = X.clone(); changed_X[0, 0] += 25.0
changed_feature_snapshot = feature_snapshot_descriptor(
    node_ids, changed_X, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF
)
assert canonical_fingerprint(changed_feature_snapshot) != FEATURE_SNAPSHOT_FINGERPRINT
try:
    validate_artifact(artifact, model.state_dict(), GRAPH_FINGERPRINT,
                      FEATURE_SCHEMA_ID, changed_feature_snapshot)
    raise AssertionError("特征内容改变后仍通过 artifact 校验")
except ValueError as exc:
    assert "特征快照" in str(exc)


## 11. 推理入口与 fail-closed

调用者只能给稳定节点 ID，不能上传邻接矩阵、覆盖 tenant 或挑选另一份 feature schema。服务从内部注册表取得授权快照与模型 bundle，验证后执行整图 forward，再只返回允许的节点。未知节点与跨 tenant ID 都拒绝；生产还需限制批量大小、审计 principal，并避免把高基数 node ID 作为监控标签。

每次推理都从实际有序特征重新计算快照摘要，并把快照 ID、时间和内容指纹写入 trace。这样同一 artifact/graph fingerprint 不可能在特征静默变化后继续返回不可追溯结果。


In [ ]:
def predict_known_nodes(auth: AuthContext, requested_ids: list[str], model: nn.Module, artifact: dict):
    auth.require("model:predict")
    if auth.tenant != artifact.get("tenant"):
        raise PermissionError("tenant 与制品不匹配")
    current_feature_snapshot = feature_snapshot_descriptor(
        node_ids, X, FEATURE_SNAPSHOT_ID, FEATURE_SNAPSHOT_AS_OF
    )
    validate_artifact(artifact, model.state_dict(), GRAPH_FINGERPRINT,
                      FEATURE_SCHEMA_ID, current_feature_snapshot)
    index = {node: i for i, node in enumerate(node_ids)}
    if not requested_ids or any(node not in index for node in requested_ids):
        raise PermissionError("请求含未知或越权节点")
    model.eval()
    with torch.no_grad():
        probs = model(X, A_hat).softmax(dim=1)
    rows = [{"node_id": node, "class_id": int(probs[index[node]].argmax()),
             "probabilities": probs[index[node]].tolist()} for node in requested_ids]
    return {"predictions": rows, "trace": {
        "artifact_id": artifact["artifact_id"],
        "graph_fingerprint": GRAPH_FINGERPRINT,
        "feature_snapshot_id": current_feature_snapshot["snapshot_id"],
        "feature_snapshot_as_of": current_feature_snapshot["as_of"],
        "feature_snapshot_fingerprint": canonical_fingerprint(current_feature_snapshot),
        "tenant": auth.tenant, "transductive": True,
    }}

served = predict_known_nodes(auth_a, node_ids[:2], model, artifact)
assert len(served["predictions"]) == 2
assert served["trace"]["artifact_id"] == artifact["artifact_id"]
assert served["trace"]["feature_snapshot_fingerprint"] == FEATURE_SNAPSHOT_FINGERPRINT
assert all(abs(sum(row["probabilities"]) - 1.0) < 1e-6 for row in served["predictions"])
try:
    predict_known_nodes(auth_a, ["tenant-b:x0"], model, artifact)
    raise AssertionError("跨 tenant 节点未被拒绝")
except PermissionError:
    pass


## 12. 复杂度、生产替换点与验收边界

本例密集邻接每层约为 (O(N^2F))、内存 (O(N^2))，只适合小图公式教学。生产应改为稀疏 COO/CSR 的 `sparse.mm`、图分区、邻居/子图采样、特征存储、离线训练与在线服务分层；同时固化 tenant/time 快照、节点顺序、随机种子、模型签名、灰度和回滚。

上线门槛包括：真实时间切分、多随机种子和置信区间、0-hop/MLP 基线、类别不平衡指标、过平滑与度数切片、OOM/超时策略、动态图重算、数据删除、模型漂移与标签延迟监控。本 Notebook 的高分不能外推为真实泛化。

In [ ]:
# 汇总式验收：避免只凭最后一个 accuracy 判断实现正确。
assert isinstance(model.conv1, GCNLayer) and isinstance(model.conv2, GCNLayer)
assert model.conv1.weight.grad is not None and model.conv2.weight.grad is not None
assert state_dict_fingerprint(model.state_dict()) == artifact["state_dict_fingerprint"]
assert GRAPH_FINGERPRINT == canonical_fingerprint(graph_payload)
assert FEATURE_SCHEMA["order"] == ["http_ratio","batch_ratio","cpu_norm","error_ratio","bias"]
assert train_seconds < 20 and max_abs_error < 2e-6
assert not any("tenant-b" in node for node in artifact["node_order"])
print({"status": "PASS", "model": "GCN-from-scratch", "best_epoch": best_epoch,
       "test_accuracy": round(test_accuracy, 3), "params": expected_params, "seconds": round(train_seconds, 3)})
assert updated_parameter_keys and restored_train_eval_loss < initial_train_eval_loss
assert artifact["feature_snapshot_fingerprint"] == canonical_fingerprint(FEATURE_SNAPSHOT)
assert served["trace"]["feature_snapshot_id"] == FEATURE_SNAPSHOT_ID


## 13. 原始与官方资料

- Kipf & Welling, *Semi-Supervised Classification with Graph Convolutional Networks*：https://arxiv.org/abs/1609.02907
- PyTorch 官方 `nn.Module` 文档：https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch 官方 Module/State 文档：https://docs.pytorch.org/docs/stable/notes/modules.html
- PyTorch 官方初始化 API：https://docs.pytorch.org/docs/stable/nn.init.html

论文用于公式和 transductive 任务背景；PyTorch 官方资料用于参数注册、forward 与 state 管理。本 Notebook 自行补充 tenant、时间、指纹、推理与失败策略，不声称复现论文完整实验。